# SISB110 — Ciência dos Dados e Inteligência Empresarial (UFAL)
## Projeto Integrador: Análise Descritiva, Inferencial e Relatório de BI sobre Saúde Mental Estudantil

**Estudante:** Bruno Martins Gomes  
**Disciplina:** SISB110 — Ciência dos Dados e Inteligência Empresarial (Sistemas de Informação / UFAL)  
**Dataset:** *Student Depression Dataset* ([Kaggle - hopesb](https://www.kaggle.com/datasets/hopesb/student-depression-dataset), Licença Apache 2.0; espelho Hugging Face `urmah/student-depression-analysis`)  
**Contexto de Negócio:** Business Intelligence para Gestão Universitária e Assistência Estudantil  

---

### Sumário do Notebook
1. **Etapa I: Problema de Negócio e Questões Decisórias (P1 a P5)**
2. **Etapa II: Tratamento, Limpeza, Tipagem e Análise Descritiva**
3. **Etapa III: Análise Inferencial, Intervalos de Confiança e Testes de Hipóteses**
4. **Etapa IV: Recomendações Práticas de BI e Limitações Metodológicas**
5. **Apêndice Metodológico: Dicionário e Tipagem das Variáveis**

## Etapa I: Definição do Problema e Formulação de Perguntas de Negócio

A evasão e o adoecimento psíquico na educação superior desafiam a sustentabilidade acadêmica e a alocação de recursos da assistência estudantil. Sob a perspectiva de **Business Intelligence (BI) para Gestão Universitária**, este estudo investiga como determinantes de sobrecarga acadêmica, hábitos de sono, vulnerabilidade financeira e satisfação estão **associados** à ocorrência de indicativo de depressão autorrelatada.

### Perguntas de Negócio e Hipóteses Formais:

* **P1 (Estimativa da Prevalência):** Qual é a proporção estimada de estudantes com indicativo de depressão na população de origem da pesquisa, e qual a margem de erro associada?  
  *Procedimento:* Intervalo de Confiança de 95% pelo método de Wilson score para proporção binomial.

* **P2 (Sobrecarga Acadêmica vs. Depressão):** O nível de pressão acadêmica percebida difere significativamente entre estudantes com e sem indicativo de depressão?  
  * $H_0$: A distribuição de pressão acadêmica percebida é idêntica entre estudantes com e sem indicativo de depressão ($F_{\text{com}}(x) = F_{\text{sem}}(x)$).  
  * $H_1$: A distribuição de pressão acadêmica percebida difere entre estudantes com e sem indicativo de depressão ($F_{\text{com}}(x) \neq F_{\text{sem}}(x)$).  
  *Procedimento:* Teste U de Mann-Whitney (principal) + Teste t de Welch (confirmatório) + Correlação Bisserial por Postos ($r_{rb}$) + Probabilidade de Superioridade (aproximadamente 77%, contando empates pela metade) + $d$ de Cohen.

* **P3 (Higiene do Sono vs. Depressão):** A duração habitual do sono está associada à ocorrência de indicativo de depressão entre os estudantes?  
  * $H_0$: A duração habitual do sono e o indicativo de depressão são independentes na população amostrada.  
  * $H_1$: A duração habitual do sono e o indicativo de depressão estão associados na população amostrada.  
  *Procedimento:* Teste Qui-Quadrado de Independência ($\chi^2$) + $V$ de Cramér + Proporção de depressão por faixa (sem teste de tendência, devido à não-monotonicidade).

* **P4 (Vulnerabilidade Financeira vs. Depressão):** A prevalência de depressão varia e exibe tendência monotônica crescente em função do nível de estresse financeiro enfrentado pelo estudante?  
  * $H_0$: A prevalência de indicativo de depressão é independente do nível de estresse financeiro e não apresenta tendência monotônica.  
  * $H_1$: A prevalência de indicativo de depressão varia e exibe tendência linear/monotônica crescente conforme o estresse financeiro aumenta.  
  *Procedimento:* Teste Qui-Quadrado de Independência ($\chi^2$) + $V$ de Cramér (efeito moderado) + Teste de Tendência de Cochran-Armitage (efeito de tendência linear forte).

* **P5 (Satisfação com Estudos vs. Rendimento Acadêmico):** Existe correlação entre a satisfação do estudante com os estudos (`Study Satisfaction`) e o seu rendimento acadêmico acumulado (`CGPA`)?  
  * $H_0$: O coeficiente de correlação monotônica populacional entre satisfação com os estudos e CGPA é nulo ($\rho = 0$).  
  * $H_1$: O coeficiente de correlação monotônica populacional entre satisfação com os estudos e CGPA é diferente de zero ($\rho \neq 0$).  
  *Procedimento:* Coeficiente de Correlação Monotônica de Spearman ($\rho$) e Linear de Pearson ($r$), acompanhados de Gráfico de Dispersão com jitter e linha de tendência linear.

In [1]:
# 1. Configuração do Ambiente e Importação de Bibliotecas
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.proportion import proportion_confint
import matplotlib.pyplot as plt
import seaborn as sns

# Reprodutibilidade e diretórios relativos
np.random.seed(42)
os.makedirs('outputs', exist_ok=True)

# Estilo gráfico padronizado
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

print("Ambiente Python inicializado com sucesso e pasta 'outputs/' pronta!")

Ambiente Python inicializado com sucesso e pasta 'outputs/' pronta!


In [2]:
# 2. Carga dos Dados Brutos a partir da Raiz do Projeto (Caminho Relativo)
data_path = 'data/Student_Depression_Dataset.csv'
df_raw = pd.read_csv(data_path)
print(f"Volume bruto carregado: {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas.")
print(f"Nulos identificados por coluna:\n{df_raw.isnull().sum()[df_raw.isnull().sum() > 0]}")

Volume bruto carregado: 27901 linhas e 18 colunas.
Nulos identificados por coluna:
Financial Stress    3
dtype: int64


### Diagnóstico Prévio de Outliers, Zeros e Suposições da Amostra

1. **Idade (`Age`):** Varia de 18 a 59 anos (mediana 25 anos, IQR = 9 anos). Pelo critério de Tukey ($Q3 + 1,5 \times \text{IQR} = 43,5$ anos), existem 12 casos ($0,04\%$) acima do limite. A manutenção das idades até 59 anos decorre da hipótese de que possam corresponder a estudantes de cursos de pós-graduação ou de ingresso tardio; **trata-se de uma possível explicação, não verificável com esta base de dados**.
2. **Horas diárias (`Work/Study Hours`):** Variam de 0 a 12 horas. Exatamente 1.700 estudantes ($6,09\%$) registraram 0 horas. Cogita-se o preenchimento durante períodos de férias, trancamento ou entressafra de disciplinas; **trata-se, igualmente, de uma possível explicação, não verificável com esta base de dados**.
3. **Zeros nas Escalas Likert:** Identificamos 9 zeros em `Academic Pressure` ($0,03\%$) e 10 zeros em `Study Satisfaction` ($0,04\%$). Como essas variáveis utilizam escalas de 1 a 5 (conforme padronizado em `Financial Stress`), esses zeros refletem código de não-resposta ou ruído. Foram tratados sob o mesmo critério padronizado: mantidos na base geral, com sensibilidade comprovando impacto desprezível na média ($< 0,0011$) e nulo na mediana ($3,0$).
4. **Zeros em `CGPA`:** 9 estudantes registraram nota $0,0$ ($0,03\%$), enquanto o percentil 1% é 5,11. Foram tratados como notas não lançadas/inválidas e excluídos das análises específicas de rendimento acadêmico.

In [3]:
# 3. Pipeline de Limpeza com Contabilidade Rigorosa
initial_n = len(df_raw)

# A. Exclusão de nulos em Financial Stress (3 linhas: IDs 22377, 68910, 97610)
df_clean = df_raw.dropna(subset=['Financial Stress']).copy()

# B. Filtragem de estudantes universitários (exclusão de 31 profissionais não-estudantes)
df_clean = df_clean[df_clean['Profession'] == 'Student'].copy()

# C. Exclusão de categoria 'Others' em Sono (18 casos) e Dieta (12 casos)
df_clean = df_clean[~df_clean['Sleep Duration'].isin(['Others'])].copy()
df_clean = df_clean[~df_clean['Dietary Habits'].isin(['Others'])].copy()

# D. Descarte de colunas quase constantes em estudantes
df_clean = df_clean.drop(columns=['Work Pressure', 'Job Satisfaction']).copy()

final_n = len(df_clean)
discarded = initial_n - final_n
print(f"Volume após limpeza: {final_n} estudantes.")
print(f"Total de linhas descartadas: {discarded} ({discarded / initial_n * 100:.2f}% da base).")

# Salvando base higienizada em data/
df_clean.to_csv('data/student_depression_cleaned.csv', index=False)
print("Arquivo 'data/student_depression_cleaned.csv' salvo com sucesso!")

Volume após limpeza: 27837 estudantes.
Total de linhas descartadas: 64 (0.23% da base).
Arquivo 'data/student_depression_cleaned.csv' salvo com sucesso!


In [4]:
# 4. Análise de Sensibilidade e Tabelas Descritivas da Etapa II
# Sensibilidade CGPA
cgpa_all = df_clean['CGPA']
cgpa_valid = df_clean[df_clean['CGPA'] > 0]['CGPA']

df_sens = pd.DataFrame({
    'Cenário': ['Com zeros (N=27.837)', 'Sem zeros (N=27.828)', 'Diferença Absoluta'],
    'Média': [cgpa_all.mean(), cgpa_valid.mean(), abs(cgpa_all.mean() - cgpa_valid.mean())],
    'Desvio Padrão': [cgpa_all.std(), cgpa_valid.std(), abs(cgpa_all.std() - cgpa_valid.std())],
    'Mediana': [cgpa_all.median(), cgpa_valid.median(), abs(cgpa_all.median() - cgpa_valid.median())],
    'IQR': [cgpa_all.quantile(0.75) - cgpa_all.quantile(0.25), cgpa_valid.quantile(0.75) - cgpa_valid.quantile(0.25), 0.0]
})
df_sens.to_csv('outputs/tabela_sensibilidade_cgpa.csv', index=False)
print("=== Tabela de Sensibilidade do CGPA ===")
print(df_sens.to_string(index=False))

# Descritiva Numéricas
num_cols = ['Age', 'CGPA', 'Academic Pressure', 'Study Satisfaction', 'Work/Study Hours', 'Financial Stress']
desc_rows = []
for col in num_cols:
    s = df_clean[col] if col != 'CGPA' else cgpa_valid
    desc_rows.append({
        'Variável': col,
        'Média': s.mean(),
        'Desvio Padrão': s.std(),
        'Mediana': s.median(),
        'IQR': s.quantile(0.75) - s.quantile(0.25),
        'Mínimo': s.min(),
        'Máximo': s.max(),
        'Assimetria': s.skew(),
        'Curtose': s.kurtosis()
    })
df_desc_num = pd.DataFrame(desc_rows)
df_desc_num.to_csv('outputs/tabela_descritiva_numericas.csv', index=False)
print("\n=== Tabela Descritiva das Variáveis Numéricas ===")
print(df_desc_num[['Variável', 'Média', 'Desvio Padrão', 'Mediana', 'IQR', 'Assimetria']].to_string(index=False))

# Bivariada Sono vs Depressão
ct_sleep = pd.crosstab(df_clean['Sleep Duration'], df_clean['Depression'], margins=True)
ct_sleep['Taxa Depressão (%)'] = (ct_sleep[1] / ct_sleep['All']) * 100
ct_sleep.to_csv('outputs/tabela_bivariada_sleep_depression.csv')

# Bivariada Estresse Financeiro vs Depressão
ct_fin = pd.crosstab(df_clean['Financial Stress'].astype(int), df_clean['Depression'], margins=True)
ct_fin['Taxa Depressão (%)'] = (ct_fin[1] / ct_fin['All']) * 100
ct_fin.to_csv('outputs/tabela_bivariada_fin_depression.csv')

=== Tabela de Sensibilidade do CGPA ===
             Cenário    Média  Desvio Padrão  Mediana  IQR
Com zeros (N=27.837) 7.656175       1.470791     7.77 2.63
Sem zeros (N=27.828) 7.658651       1.464569     7.77 2.63
  Diferença Absoluta 0.002476       0.006222     0.00 0.00

=== Tabela Descritiva das Variáveis Numéricas ===
          Variável     Média  Desvio Padrão  Mediana  IQR  Assimetria
               Age 25.819772       4.906603    25.00 9.00    0.132930
              CGPA  7.658651       1.464569     7.77 2.63   -0.073875
 Academic Pressure  3.140748       1.381791     3.00 2.00   -0.134912
Study Satisfaction  2.943385       1.361073     3.00 2.00    0.011021
  Work/Study Hours  7.158889       3.706690     8.00 6.00   -0.456167
  Financial Stress  3.139814       1.437217     3.00 2.00   -0.130231


In [5]:
# 5. Visualizações Gráficas da Etapa II (Alta Resolução em outputs/)
# 1. Distribuições Numéricas
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
colors_palette = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c', '#9b59b6', '#34495e']
for idx, col in enumerate(num_cols):
    ax = axes[idx // 3, idx % 3]
    s = df_clean[col] if col != 'CGPA' else cgpa_valid
    sns.histplot(s, kde=True, ax=ax, color=colors_palette[idx], discrete=(col not in ['Age', 'CGPA']))
    ax.set_title(f'Distribuição de {col}', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/01_distribuicoes_numericas.png', dpi=300)
plt.close()

# 2. Sono vs Depressão
sleep_order = ['Less than 5 hours', '5-6 hours', '7-8 hours', 'More than 8 hours']
sleep_rates = [ct_sleep.loc[s, 'Taxa Depressão (%)'] for s in sleep_order]
fig, ax = plt.subplots(figsize=(8, 4.8))
bars = ax.bar(['< 5 hours', '5-6 hours', '7-8 hours', '> 8 hours'], sleep_rates, color=['#e74c3c', '#e67e22', '#2ecc71', '#3498db'], width=0.55)
ax.axhline(58.52, color='#555555', linestyle='--', label='Média Global (58,52%)')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2, f'{bar.get_height():.1f}%', ha='center', fontweight='bold')
ax.set_ylim(0, 80)
ax.set_ylabel('Taxa com Indicativo de Depressão (%)')
ax.set_title('Taxa de Depressão por Faixa Habitual de Sono', fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('outputs/02_sleep_vs_depression.png', dpi=300)
plt.close()

# 3. Pressão Acadêmica por Depressão
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(x='Depression', y='Academic Pressure', data=df_clean, ax=ax, palette=['#3498db', '#e74c3c'], width=0.4)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Sem Depressão (0)', 'Com Depressão (1)'])
ax.set_title('Pressão Acadêmica Percebida por Status de Depressão', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/03_academic_pressure_by_depression.png', dpi=300)
plt.close()

# 4. Heatmaps de Correlação
sub_num = df_clean[['Age', 'CGPA', 'Academic Pressure', 'Study Satisfaction', 'Work/Study Hours', 'Financial Stress']].copy()
sub_num.loc[sub_num['CGPA'] == 0, 'CGPA'] = np.nan
corr_pearson = sub_num.corr(method='pearson')
corr_spearman = sub_num.corr(method='spearman')
corr_pearson.to_csv('outputs/matriz_correlacao_pearson.csv')
corr_spearman.to_csv('outputs/matriz_correlacao_spearman.csv')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(corr_pearson, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[0], square=True)
axes[0].set_title('Correlação Linear de Pearson (r)', fontweight='bold')
sns.heatmap(corr_spearman, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1], square=True)
axes[1].set_title('Correlação Monotônica de Spearman (rho)', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/05_correlation_matrices.png', dpi=300)
plt.close()

print("Gráficos 01, 02, 03 e 05 salvos com sucesso em 'outputs/'!")

Gráficos 01, 02, 03 e 05 salvos com sucesso em 'outputs/'!


In [6]:
# 6. Etapa III: Procedimentos Inferenciais e Testes de Hipóteses (Alfa = 0.05)
N = len(df_clean)

# --- P1: IC de Wilson 95% para Depressão ---
k_dep = (df_clean['Depression'] == 1).sum()
p_hat = k_dep / N
ci_low_w, ci_upp_w = proportion_confint(k_dep, N, alpha=0.05, method='wilson')
print(f"P1 - Proporção Amostral: {p_hat:.4f} ({p_hat*100:.2f}%)")
print(f"P1 - IC 95% de Wilson: [{ci_low_w*100:.2f}%, {ci_upp_w*100:.2f}%] (Margem de erro: ±{(ci_upp_w - ci_low_w)/2*100:.2f}%)")

# --- P2: Academic Pressure vs Depression ---
ap_sem = df_clean.loc[df_clean['Depression'] == 0, 'Academic Pressure']
ap_com = df_clean.loc[df_clean['Depression'] == 1, 'Academic Pressure']
u_stat, p_mann = stats.mannwhitneyu(ap_com, ap_sem, alternative='two-sided')
r_rb = (2 * u_stat - (len(ap_com) * len(ap_sem))) / (len(ap_com) * len(ap_sem))
prob_sup = (r_rb + 1) / 2  # Probabilidade de Superioridade (CLES)

t_stat, p_welch = stats.ttest_ind(ap_com, ap_sem, equal_var=False)
s_pooled = np.sqrt(((len(ap_com)-1)*ap_com.std()**2 + (len(ap_sem)-1)*ap_sem.std()**2)/(N-2))
cohen_d = (ap_com.mean() - ap_sem.mean()) / s_pooled

print(f"\nP2 - Mann-Whitney U: {u_stat:.1f}, p-valor = {p_mann:.4e}")
print(f"P2 - Correlação Bisserial por Postos (r_rb): {r_rb:.4f}")
print(f"P2 - Probabilidade de Superioridade: {prob_sup*100:.2f}% (probabilidade de superioridade de aproximadamente 77%, contando empates pela metade)")
print(f"P2 - Welch t: {t_stat:.2f}, p-valor = {p_welch:.4e}, d de Cohen: {cohen_d:.4f} (Efeito Grande)")

# --- P3: Sleep Duration vs Depression ---
ct_sleep_test = pd.crosstab(df_clean['Sleep Duration'], df_clean['Depression'])
chi2_s, p_chi_s, dof_s, _ = stats.chi2_contingency(ct_sleep_test)
v_cramer_s = np.sqrt(chi2_s / (N * (min(ct_sleep_test.shape) - 1)))
print(f"\nP3 - Qui-Quadrado Sono: chi2 = {chi2_s:.2f}, gl = {dof_s}, p-valor = {p_chi_s:.4e}")
print(f"P3 - V de Cramér: {v_cramer_s:.4f} (Associação Fraca, ~0.10, não-monotônica)")

# --- P4: Financial Stress vs Depression ---
ct_fin_test = pd.crosstab(df_clean['Financial Stress'].astype(int), df_clean['Depression'])
chi2_f, p_chi_f, dof_f, _ = stats.chi2_contingency(ct_fin_test)
v_cramer_f = np.sqrt(chi2_f / (N * (min(ct_fin_test.shape) - 1)))

scores = np.array([1, 2, 3, 4, 5])
n_i = ct_fin_test.sum(axis=1).values
k_i = ct_fin_test[1].values
p_bar = np.sum(k_i) / N
w_bar = np.sum(n_i * scores) / N
T = np.sum(scores * (k_i - n_i * p_bar))
var_T = p_bar * (1 - p_bar) * np.sum(n_i * (scores - w_bar)**2)
z_ca = T / np.sqrt(var_T)
p_ca = 2 * (1 - stats.norm.cdf(abs(z_ca)))

print(f"\nP4 - Qui-Quadrado Finanças: chi2 = {chi2_f:.2f}, gl = {dof_f}, p-valor = {p_chi_f:.4e}")
print(f"P4 - V de Cramér: {v_cramer_f:.4f} (Associação Moderada)")
print(f"P4 - Tendência de Cochran-Armitage: Z = {z_ca:.2f}, p-valor = {p_ca:.4e} (Tendência Monotônica Forte)")

# --- P5: Study Satisfaction vs CGPA ---
cgpa_clean = df_clean.loc[df_clean['CGPA'] > 0, 'CGPA']
sat_clean = df_clean.loc[df_clean['CGPA'] > 0, 'Study Satisfaction']
rho, p_rho = stats.spearmanr(sat_clean, cgpa_clean)
r, p_r = stats.pearsonr(sat_clean, cgpa_clean)
print(f"\nP5 - Spearman: rho = {rho:.4f}, p-valor = {p_rho:.4e} (|rho| < 0.10, Efeito Prático Desprezível)")
print(f"P5 - Pearson:  r = {r:.4f}, p-valor = {p_r:.4e}")

P1 - Proporção Amostral: 0.5852 (58.52%)
P1 - IC 95% de Wilson: [57.94%, 59.10%] (Margem de erro: ±0.58%)

P2 - Mann-Whitney U: 144916287.5, p-valor = 0.0000e+00
P2 - Correlação Bisserial por Postos (r_rb): 0.5408
P2 - Probabilidade de Superioridade: 77.04% (probabilidade de superioridade de aproximadamente 77%, contando empates pela metade)
P2 - Welch t: 89.21, p-valor = 0.0000e+00, d de Cohen: 1.0950 (Efeito Grande)

P3 - Qui-Quadrado Sono: chi2 = 277.90, gl = 3, p-valor = 6.0295e-60
P3 - V de Cramér: 0.0999 (Associação Fraca, ~0.10, não-monotônica)

P4 - Qui-Quadrado Finanças: chi2 = 3695.04, gl = 4, p-valor = 0.0000e+00
P4 - V de Cramér: 0.3643 (Associação Moderada)
P4 - Tendência de Cochran-Armitage: Z = 60.67, p-valor = 0.0000e+00 (Tendência Monotônica Forte)

P5 - Spearman: rho = -0.0467, p-valor = 6.8498e-15 (|rho| < 0.10, Efeito Prático Desprezível)
P5 - Pearson:  r = -0.0466, p-valor = 7.5185e-15


In [7]:
# 7. Gráfico de Dispersão: Study Satisfaction vs. CGPA (Exigência do Roteiro - P5)
valid_cgpa = df_clean[df_clean['CGPA'] > 0].copy()

# Jitter horizontal leve para permitir visualização da densidade em escala discreta (0 a 5)
np.random.seed(42)
jitter = np.random.uniform(-0.15, 0.15, size=len(valid_cgpa))
valid_cgpa['Study_Satisfaction_Jitter'] = valid_cgpa['Study Satisfaction'] + jitter

plt.figure(figsize=(8, 5))
plt.scatter(
    valid_cgpa['Study_Satisfaction_Jitter'],
    valid_cgpa['CGPA'],
    alpha=0.15,
    color='#2980b9',
    s=12,
    edgecolors='none',
    label='Estudantes (com jitter horizontal)'
)

# Linha de tendência linear por regressão
m, b = np.polyfit(valid_cgpa['Study Satisfaction'], valid_cgpa['CGPA'], 1)
x_vals = np.array([0, 5])
plt.plot(x_vals, m * x_vals + b, color='#c0392b', linewidth=2.2,
         label=f'Tendência Linear (ρ = {rho:.3f}, p < 0,001)')

plt.title('Gráfico de Dispersão: Satisfação com os Estudos × Rendimento (CGPA)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Satisfação com os Estudos (Escala 0 a 5)', fontsize=10)
plt.ylabel('Rendimento Acadêmico Acumulado (CGPA: 0 a 10)', fontsize=10)
plt.xticks([0, 1, 2, 3, 4, 5], ['0', '1', '2', '3', '4', '5'])
plt.ylim(4.5, 10.5)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True, facecolor='white', framealpha=0.9, loc='lower right', fontsize=9)
plt.tight_layout()

output_scatter = 'outputs/04_dispersao_satisfacao_cgpa.png'
plt.savefig(output_scatter, dpi=300)
plt.close()
print(f"Gráfico de dispersão salvo com sucesso em '{output_scatter}' (N = {len(valid_cgpa)}, rho = {rho:.4f})!")

Gráfico de dispersão salvo com sucesso em 'outputs/04_dispersao_satisfacao_cgpa.png' (N = 27828, rho = -0.0467)!


In [8]:
# 8. Controle de Comparações Múltiplas (4 Famílias de Hipóteses)
# P1 é IC de estimativa; P2 possui dois testes concorrentes sobre a mesma hipótese.
hypothesis_families = [
    ('Família 1: Pressão Acadêmica vs Depressão (P2 - Mann-Whitney)', p_mann),
    ('Família 2: Duração do Sono vs Depressão (P3 - Qui-Quadrado)', p_chi_s),
    ('Família 3: Estresse Financeiro vs Depressão (P4 - Qui-Quadrado / Cochran-Armitage)', min(p_chi_f, p_ca)),
    ('Família 4: Satisfação com Estudos vs CGPA (P5 - Spearman)', p_rho)
]
hyp_sorted = sorted(hypothesis_families, key=lambda x: x[1])
m_fam = len(hyp_sorted)

multi_res = []
for i, (name, p_val) in enumerate(hyp_sorted):
    alpha_holm = 0.05 / (m_fam - i)
    alpha_bonf = 0.05 / m_fam
    multi_res.append({
        'Família de Hipótese': name,
        'p-valor Bruto': f'{p_val:.4e}',
        'Limiar Holm': f'{alpha_holm:.5f}',
        'Limiar Bonferroni': f'{alpha_bonf:.5f}',
        'Rejeita H0 após Controle?': p_val < alpha_holm
    })
df_multi = pd.DataFrame(multi_res)
print("=== Controle de Comparações Múltiplas por Família de Hipótese ===")
print(df_multi.to_string(index=False))

=== Controle de Comparações Múltiplas por Família de Hipótese ===
                                                               Família de Hipótese p-valor Bruto Limiar Holm Limiar Bonferroni  Rejeita H0 após Controle?
                     Família 1: Pressão Acadêmica vs Depressão (P2 - Mann-Whitney)    0.0000e+00     0.01250           0.01250                       True
Família 3: Estresse Financeiro vs Depressão (P4 - Qui-Quadrado / Cochran-Armitage)    0.0000e+00     0.01667           0.01250                       True
                       Família 2: Duração do Sono vs Depressão (P3 - Qui-Quadrado)    6.0295e-60     0.02500           0.01250                       True
                         Família 4: Satisfação com Estudos vs CGPA (P5 - Spearman)    6.8498e-15     0.05000           0.01250                       True


## Etapa IV: Storytelling, Recomendações Práticas e Limitações Metodológicas

### Síntese dos Achados de Business Intelligence
1. **Estresse Financeiro como Preditador Associativo Mais Expressivo:** A prevalência de indicativo de depressão sobe de forma estritamente monotônica de **31,8%** no estrato mais baixo para **81,2%** no estrato de vulnerabilidade financeira máxima ($V = 0,36$; tendência $Z = 60,67$).
2. **Sobrecarga Acadêmica:** Estudantes com depressão reportam pressão percebida substantivamente maior ($3,69$ vs $2,36$, $d = 1,10$). A métrica de efeito indica uma **probabilidade de superioridade de aproximadamente 77%, contando empates pela metade** (ajuste metodológico devido à escala discreta com múltiplos empates).
3. **Higiene do Sono e Risco nos Extremos:** Estudantes que dormem menos de 5 horas apresentam a taxa mais elevada ($64,5%$) e mais de 8 horas a menor ($50,9%$), embora a relação global seja fraca ($V = 0,10$) e não-monotônica.
4. **Desconexão entre Notas e Satisfação (P5):** Conforme demonstrado visualmente no Gráfico de Dispersão salvo em `outputs/04_dispersao_satisfacao_cgpa.png`, a nuvem de pontos com jitter e a inclinação nula da linha de tendência linear ilustram a ausência de relevância prática entre o rendimento acadêmico (CGPA) e a satisfação com os estudos ($\rho = -0,047$).

### Recomendações e Plano de Ação Institucional
* **Ação 1 (Prioridade Orçamentária e Permanência):** Focalizar auxílios e bolsas de permanência nos estudantes com maior estresse financeiro.
* **Ação 2 (Gestão Pedagógica):** Mitigar picos de sobrecarga curricular e rever calendários de avaliações concentradas.
* **Ação 3 (Campanhas de Bem-Estar e Sono):** Promover oficinas de manejo de tempo e higiene do sono.
* **Ação 4 (Acolhimento sem Viés de Nota):** Desvincular atendimentos psicopedagógicos de critérios de coeficiente de rendimento.

### Limitações Metodológicas e Rigor Ético
- **Associação vs. Causalidade:** Dados observacionais de corte transversal; risco de causalidade reversa (a depressão pode elevar a percepção de estresse e sobrecarga).
- **Autorrelato e Ausência de Diagnóstico Clínico:** A classificação baseia-se em questionário autoaplicado, e não em entrevista psiquiátrica clínica estruturada (DSM-5 / CID-11).
- **Viés de Amostra:** As taxas elevadas (58,5% de depressão e 63,3% de ideação suicida) sinalizam forte viés de autosseleção na pesquisa.
- **Cuidado Ético:** A variável de ideação suicida deve ser tratada como indicador de alerta institucional para acolhimento e suporte psicossocial, e nunca para rotulação estigmatizante.

## Apêndice Metodológico: Classificação e Tipagem de Variáveis

| Nome da Coluna | Tipo Estatístico | Escala / Domínio | Tratamento Aplicado |
|---|---|---|---|
| `id` | Identificador Técnico | Inteiro único | Mantido para rastreabilidade; 0 duplicatas |
| `Gender` | Categórica Nominal | Dicotômica ('Male', 'Female') | Tipada como categoria dicotômica |
| `Age` | Quantitativa Discreta | Anos completos (18 a 59) | Mantidos outliers naturais válidos sob suposição discente |
| `City` | Categórica Nominal | 52 cidades | Tipada como texto/categoria |
| `Profession` | Categórica Nominal | Texto ('Student') | Filtrada para apenas estudantes (31 não-estudantes removidos) |
| `Academic Pressure` | Qualitativa Ordinal / Escala | Discreta (0 a 5) | Mantida na escala ordinal (9 zeros analisados por sensibilidade) |
| `CGPA` | Quantitativa Contínua/Razão | Escala decimal (0,0 a 10,0) | 9 zeros tratados como inválidos nas análises de CGPA |
| `Study Satisfaction` | Qualitativa Ordinal / Escala | Discreta (0 a 5) | Mantida na escala ordinal (10 zeros analisados por sensibilidade) |
| `Sleep Duration` | Categórica Ordinal | 4 faixas (< 5h, 5-6h, 7-8h, > 8h) | 'Others' (18 casos) excluído por ruído de codificação |
| `Dietary Habits` | Categórica Ordinal | 3 níveis (Unhealthy, Moderate, Healthy) | 'Others' (12 casos) excluído por ruído de codificação |
| `Degree` | Categórica Nominal | 28 títulos acadêmicos | Tipada como categórica |
| `Have you ever had suicidal thoughts ?` | Categórica Nominal | Dicotômica ('Yes', 'No') | Tipada como categórica binária de alerta institucional |
| `Work/Study Hours` | Quantitativa Discreta | Horas inteiras/dia (0 a 12) | Mantida (6,09% com 0 horas analisados sob suposição de férias) |
| `Financial Stress` | Qualitativa Ordinal / Escala | Discreta (1 a 5) | 3 nulos excluídos por complete case analysis |
| `Family History of Mental Illness` | Categórica Nominal | Dicotômica ('Yes', 'No') | Tipada como categórica binária |
| `Depression` | Categórica Nominal / Binária | Dicotômica (0 = Não, 1 = Sim) | Desfecho principal de interesse analítico |